### dependencies

In [3]:
import openai
from qdrant_client import QdrantClient

In [4]:
def getEmbedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding

In [5]:
qdrant_client = QdrantClient(host="localhost", port=6333)

In [7]:
### Retrieval function
def retrieve_data(query, k=5):
    query_embedding = getEmbedding(query)
    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_contexts_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_contexts_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_contexts_ratings": retrieved_contexts_ratings
    }

In [8]:
results = retrieve_data("wireless headphones", k=5)

In [9]:
results

{'retrieved_context_ids': ['B0C68LS4MX',
  'B0BG28SBL6',
  'B09MQ49WT9',
  'B0BY12VN83',
  'B0BZ3WYJL1'],
 'retrieved_context': ['Wireless Earbuds Bluetooth 5.3 Headphones HiFi Stereo Wireless Headphones with 4 ENC Noise Clear Canceling Mic Wireless Headset with LED Display Charging Case 42H Playtime Touch Control IP7 Waterproof Bluetooth 5.3 and One Step Pairing: J8 true wireless earbuds with the advanced Bluetooth 5.3 technology, provides faster pairing, stable connection and universal compatibility. Simply the lid open, bluetooth earbuds will connect each other automatically and then only one step easily enter mobile phone bluetooth setting to pair, you will be in your music world in a couple of seconds. Hi-Fi Stereo Sound Quality: Mysic J8 earbuds wireless headphones offers a truly natural, authentic sound and powerful bass performance with 13mm large size speaker driver- the drive area is 3.77 times than the normal drive area. Half in-ear acoustic structure and high-definition ren

### format retrived context fn

In [10]:
def process_context(context):
    formated_context = ""
    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_contexts_ratings"]):
        formated_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formated_context

In [11]:
preprocessed_context = process_context(results)
print(preprocessed_context)

- ID: B0C68LS4MX, rating: 4.7, description: Wireless Earbuds Bluetooth 5.3 Headphones HiFi Stereo Wireless Headphones with 4 ENC Noise Clear Canceling Mic Wireless Headset with LED Display Charging Case 42H Playtime Touch Control IP7 Waterproof Bluetooth 5.3 and One Step Pairing: J8 true wireless earbuds with the advanced Bluetooth 5.3 technology, provides faster pairing, stable connection and universal compatibility. Simply the lid open, bluetooth earbuds will connect each other automatically and then only one step easily enter mobile phone bluetooth setting to pair, you will be in your music world in a couple of seconds. Hi-Fi Stereo Sound Quality: Mysic J8 earbuds wireless headphones offers a truly natural, authentic sound and powerful bass performance with 13mm large size speaker driver- the drive area is 3.77 times than the normal drive area. Half in-ear acoustic structure and high-definition rendering technology, which can give you incredible sound quality and crystal crisp trebl

### create prompt template fn

In [12]:
def build_prompt(preprocessed_context, user_query):
    prompt = f"""
    you are a shopping assistant that can answeer questions about the products in stock.
    you will be given a question and a list of context.

    instructions:
    * answer the question based on the provided context only.
    * never use word context and refer to it as the available products.
    * do not use markdown formmatting in your answer.
    
    Context:
    {preprocessed_context}

    Question: {user_query}
    """
    return prompt

In [13]:
prompt = build_prompt(preprocessed_context, "what are the best wireless headphones available?")
print(prompt)


    you are a shopping assistant that can answeer questions about the products in stock.
    you will be given a question and a list of context.

    instructions:
    * answer the question based on the provided context only.
    * never use word context and refer to it as the available products.
    * do not use markdown formmatting in your answer.

    Context:
    - ID: B0C68LS4MX, rating: 4.7, description: Wireless Earbuds Bluetooth 5.3 Headphones HiFi Stereo Wireless Headphones with 4 ENC Noise Clear Canceling Mic Wireless Headset with LED Display Charging Case 42H Playtime Touch Control IP7 Waterproof Bluetooth 5.3 and One Step Pairing: J8 true wireless earbuds with the advanced Bluetooth 5.3 technology, provides faster pairing, stable connection and universal compatibility. Simply the lid open, bluetooth earbuds will connect each other automatically and then only one step easily enter mobile phone bluetooth setting to pair, you will be in your music world in a couple of seconds

### generate answer

In [16]:
def generate_answer(prompt, model="gpt-5.4-nano"):
    response = openai.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": prompt}
        ],
        reasoning_effort="none"
    )
    return response.choices[0].message.content.strip()

In [17]:
generated_answer = generate_answer(prompt)
print(generated_answer)

Based on the available products, the best overall wireless option is the Wireless Earbuds Bluetooth Headphones with 120H Playtime, IPX7 Waterproof, and a 2600mAh charging case (ID: B0BZ3WYJL1, rating 4.7). It offers the longest total playtime (up to 120 hours with the case), has IPX7 waterproof protection, includes a digital power display on the case, and uses Bluetooth 5.3 with automatic connection when the case is opened.

If you want a high-performance sound and features focused on workout durability, a close alternative is the Wireless Earbuds Bluetooth 5.3 Headphones HiFi Stereo with 4 ENC Noise Clear Mic, IP7 Waterproof, and 42H playtime (ID: B0C68LS4MX, rating 4.7).


### combined rag pl

In [18]:
def rag_pipeline(query, k=5):
    results = retrieve_data(query, k)
    preprocessed_context = process_context(results)
    prompt = build_prompt(preprocessed_context, query)
    answer = generate_answer(prompt)
    return answer

In [19]:
rag_pipeline("what are the best wireless headphones available?")

'Based on the available products, the “best” wireless option would be:\n\n1) B0C68LS4MX (rating 4.7) — Wireless Earbuds Bluetooth 5.3 with HiFi stereo sound, 4 ENC noise-canceling mic, touch controls, LED battery case display, and IP7 waterproof/sweatproof, with up to 42 hours total playtime using the charging case.\n\nIf you prefer neckband/sport wired-Bluetooth style instead:\n2) B0BG28SBL6 (rating 4.3) — Bluetooth 5.0 aptX wireless earbuds with CVC mic, IPX6 water resistance, and up to 9 hours of aptX audio performance (up to 6 hours stated as uninterrupted audio).'

In [ ]:
rag_pipeline("what are the best wireless headphones available?, get me products with 6 or above rating and are under $100")